In [1]:
import os
import json
import pandas as pd
import numpy as np

def read_jsonl_in_chunks(path, chunk_size=50):
    with open(path, 'r', encoding='utf-8') as f:
        while True:
            lines = [f.readline() for _ in range(chunk_size)]
            lines = [ln for ln in lines if ln.strip()]
            if not lines:
                break
            from io import StringIO
            chunk_df = pd.read_json(StringIO(''.join(lines)), lines=True)
            yield chunk_df

chunks = list(read_jsonl_in_chunks('data/events_lines.json', chunk_size=50))
print(f"Прочитано блоков: {len(chunks)}")
print(f"Размеры блоков: {[len(c) for c in chunks]}")
print("\nПример блока 1 :")
print(chunks[0].head())

Прочитано блоков: 40
Размеры блоков: [50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50, 50]

Пример блока 1 :
   event_id  user_id          event_time   event_type         page
0      2001     1029 2024-03-19 00:34:38     purchase  /product/20
1      2002     1056 2024-03-25 17:16:08  add_to_cart        /cart
2      2003     1081 2024-02-06 06:09:49  add_to_cart  /product/30
3      2004     1005 2024-03-20 04:35:23     purchase   /product/5
4      2005     1033 2024-02-21 09:51:53        click  /product/11


In [2]:
block_stats = []

for i, chunk in enumerate(read_jsonl_in_chunks('data/events_lines.json', chunk_size=50)):
    counts = chunk['event_type'].value_counts().rename('count').reset_index()
    counts.columns = ['event_type', 'count']
    counts['block'] = i + 1
    block_stats.append(counts)
    print(f"Блок {i+1}:")
    print(counts.to_string(index=False), "\n")

all_stats = pd.concat(block_stats, ignore_index=True)
total_stats = all_stats.groupby('event_type', as_index=False)['count'].sum()
print("Итоговая статистика по типам событий:")
print(total_stats)

Блок 1:
 event_type  count  block
   purchase     16      1
     scroll     10      1
add_to_cart      9      1
      click      8      1
  page_view      7      1 

Блок 2:
 event_type  count  block
add_to_cart     12      2
      click     11      2
  page_view     10      2
     scroll      9      2
   purchase      8      2 

Блок 3:
 event_type  count  block
  page_view     12      3
   purchase     11      3
     scroll     10      3
      click     10      3
add_to_cart      7      3 

Блок 4:
 event_type  count  block
   purchase     14      4
      click     13      4
  page_view     10      4
add_to_cart      8      4
     scroll      5      4 

Блок 5:
 event_type  count  block
   purchase     17      5
      click     13      5
  page_view      8      5
     scroll      6      5
add_to_cart      6      5 

Блок 6:
 event_type  count  block
     scroll     14      6
      click     13      6
  page_view     10      6
add_to_cart      8      6
   purchase      5      6 

Блок

In [3]:
total_stats.to_csv('data/events_stats.csv', index=False)
print(open('data/events_stats.csv').read())

event_type,count
add_to_cart,387
click,416
page_view,379
purchase,413
scroll,405



In [4]:
all_events = pd.concat(read_jsonl_in_chunks('data/events_lines.json', chunk_size=50),
                       ignore_index=True)

user_event_counts = (all_events.groupby('user_id', as_index=False)
                     .size()
                     .rename(columns={'size': 'events_count'}))

print("Всего пользователей:", len(user_event_counts))
print(user_event_counts.sort_values('events_count', ascending=False).head(10))

active_users = user_event_counts[user_event_counts['events_count'] > 10].copy()
print(f"\nАктивных пользователей (>10 событий): {len(active_users)}")
print(active_users)

orders_files = ['data/orders_2024_01.csv',
                'data/orders_2024_02.csv',
                'data/orders_2024_03.csv']
orders = pd.concat([pd.read_csv(f) for f in orders_files], ignore_index=True)
print(f"\nВсего заказов: {len(orders)}")

events_orders = active_users.merge(orders, on='user_id', how='inner')
print("\nОбъединённая таблица (активные пользователи и заказы):")
print(events_orders.head(10))
print(f"\nСтрок в результате: {len(events_orders)}")

Всего пользователей: 100
    user_id  events_count
71     1072            32
51     1052            31
95     1096            31
90     1091            31
39     1040            31
0      1001            29
17     1018            28
92     1093            28
20     1021            27
74     1075            27

Активных пользователей (>10 событий): 100
    user_id  events_count
0      1001            29
1      1002            25
2      1003            21
3      1004            22
4      1005            21
..      ...           ...
95     1096            31
96     1097            20
97     1098            19
98     1099            26
99     1100            16

[100 rows x 2 columns]

Всего заказов: 15000

Объединённая таблица (активные пользователи и заказы):
   user_id  events_count  order_id       order_datetime  product_id     price  \
0     1001            29        10  2024-01-22 19:11:48          24  13397.56   
1     1001            29       113  2024-01-31 17:38:20           4   

In [5]:
active_ids = sorted(active_users['user_id'].tolist())

with open('data/active_users.txt', 'w', encoding='utf-8') as f:
    for uid in active_ids:
        f.write(f"{uid}\n")

print("Активные:", active_ids)
print("\nСодержимое data/active_users.txt:")
print(open('data/active_users.txt').read())

Активные: [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008, 1009, 1010, 1011, 1012, 1013, 1014, 1015, 1016, 1017, 1018, 1019, 1020, 1021, 1022, 1023, 1024, 1025, 1026, 1027, 1028, 1029, 1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039, 1040, 1041, 1042, 1043, 1044, 1045, 1046, 1047, 1048, 1049, 1050, 1051, 1052, 1053, 1054, 1055, 1056, 1057, 1058, 1059, 1060, 1061, 1062, 1063, 1064, 1065, 1066, 1067, 1068, 1069, 1070, 1071, 1072, 1073, 1074, 1075, 1076, 1077, 1078, 1079, 1080, 1081, 1082, 1083, 1084, 1085, 1086, 1087, 1088, 1089, 1090, 1091, 1092, 1093, 1094, 1095, 1096, 1097, 1098, 1099, 1100]

Содержимое data/active_users.txt:
1001
1002
1003
1004
1005
1006
1007
1008
1009
1010
1011
1012
1013
1014
1015
1016
1017
1018
1019
1020
1021
1022
1023
1024
1025
1026
1027
1028
1029
1030
1031
1032
1033
1034
1035
1036
1037
1038
1039
1040
1041
1042
1043
1044
1045
1046
1047
1048
1049
1050
1051
1052
1053
1054
1055
1056
1057
1058
1059
1060
1061
1062
1063
1064
1065
1066
1067
1068
1069
1070
1071